In [60]:
from dotenv import load_dotenv
from langfuse import get_client, Evaluation
from datetime import datetime
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_qdrant import FastEmbedSparse, RetrievalMode
from getpass import getpass
import pandas as pd
import os
import re

In [73]:
load_dotenv() # TODO : pour recuperer les infos utiles a Langfuse
langfuse = get_client()

dataset_name = "dev_dataset"

In [3]:
dataset = pd.read_csv('../test/tb_panchaud_jeu_test - saskya_processing.csv', header=0) # TODO : update path

langfuse.create_dataset(
    name=dataset_name,
    description="Dataset containg dev questions (2 April 2026)",
    metadata={
        "author": "Saskya",
        "date": "2026-04-02",
        "type": "dev"
    }
)

for _, row in dataset.iterrows():
    if row['Dataset'] == "dev" and pd.notna(row["Id document"]) and pd.notna(row["Question"]):
        langfuse.create_dataset_item(
            dataset_name=dataset_name,
            input={
                "query": str(row["Question"])
            },
            metadata={
                "expected_doc_id": str(row["Id document"])
            }
        )

# TODO : si documents de nouveau telecharges alors id vont changer, comment fixer ca et mettre une reference stable dans le dataset de dev/test ???

In [67]:
COLLECTION_NAME = "tika_bge_RCTS_window_hybrid"
K = 1000

if not os.getenv("BGE_API_KEY_EMBEDDINGS"):
    os.environ["BGE_API_KEY_EMBEDDINGS"] = getpass("Enter your RCP API key for embedding model (bge): ")

embeddings = OpenAIEmbeddings(
    model="BAAI/bge-m3",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("BGE_API_KEY_EMBEDDINGS")
)

sparse = FastEmbedSparse(model_name="Qdrant/bm42-all-minilm-l6-v2-attentions")
qdrant = QdrantVectorStore.from_existing_collection(
    url="http://localhost:6333",
    embedding=embeddings,
    sparse_embedding=sparse,
    collection_name=COLLECTION_NAME,
    retrieval_mode=RetrievalMode.HYBRID
)

In [68]:
def retrieval_task(*, item, **kwargs):
    input = item.input
    query = input.get("query")
    hits = qdrant.similarity_search_with_score(query, k=K)

    retrieved_doc_ids = []
    retrieved_scores = []

    for hit in hits:
        document, score = hit
        # TODO : update si collection Qdrant modifiee
        #source = document.metadata.get("doc_id")
        #doc_id = re.search(r"^data/(.+)\.", source).group(1)
        doc_id = document.metadata.get("doc_id")
        retrieved_doc_ids.append(doc_id)
        retrieved_scores.append(score)

    return {
        "retrieved_doc_ids": retrieved_doc_ids,
        "retrieved_scores": retrieved_scores
    }

In [69]:
def hit_at_1_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    retrieved_scores = output.get("retrieved_scores")
    value = 1.0 if expected_doc_id == retrieved_doc_ids[0] else 0.0
    return Evaluation(
        name="hit_at_1",
        value=value,
        comment=f"expected_doc_id={expected_doc_id}, top1={retrieved_doc_ids[0]} with score {retrieved_scores[0]}", # TODO : mettre les scores autrement pour pouvoir average dessus ?
    )

def hit_at_5_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    retrieved_scores = output.get("retrieved_scores")
    value = 1.0 if expected_doc_id in retrieved_doc_ids[:5] else 0.0
    return Evaluation(
        name="hit_at_5",
        value=value,
        comment=f"expected_doc_id={expected_doc_id}, top5={retrieved_doc_ids[:5]} with scores {retrieved_scores[:5]}",
    )

def hit_at_20_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    retrieved_scores = output.get("retrieved_scores")
    value = 1.0 if expected_doc_id in retrieved_doc_ids[:20] else 0.0
    return Evaluation(
        name="hit_at_20",
        value=value,
        comment=f"expected_doc_id={expected_doc_id}, top20={retrieved_doc_ids[:20]} with score {retrieved_scores[:20]}", # TODO : mettre les scores autrement pour pouvoir average dessus ?
    )

def mrr_doc_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    reciprocal_rank = 0.0
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id == expected_doc_id:
            reciprocal_rank = 1.0 / rank
            break
    return Evaluation(
        name="mrr_doc",
        value=reciprocal_rank,
        comment=f"expected_doc_id={expected_doc_id}, retrieved={retrieved_doc_ids}",
    )

def nb_correct_doc_in_chunks(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    nb_correct_docs = 0
    for retrieved_doc_id in retrieved_doc_ids:
        if retrieved_doc_id == expected_doc_id:
            nb_correct_docs += 1
    return Evaluation(
        name="nb_correct_doc_in_chunks",
        value=nb_correct_docs,
        comment=f"expected_doc_id={expected_doc_id}, retrieved={retrieved_doc_ids}",
    )

In [70]:
def avg_hit_at_1(*, item_results, **kwargs):
    vals = [
        ev.value for result in item_results
        for ev in result.evaluations
        if ev.name == "hit_at_1"
    ]
    if not vals:
        return Evaluation(name="avg_hit_at_1", value=None)
    avg = sum(vals) / len(vals)
    return Evaluation(name="avg_hit_at_1", value=avg)

def avg_hit_at_5(*, item_results, **kwargs):
    vals = [
        ev.value for result in item_results
        for ev in result.evaluations
        if ev.name == "hit_at_5"
    ]
    if not vals:
        return Evaluation(name="avg_hit_at_5", value=None)
    avg = sum(vals) / len(vals)
    return Evaluation(name="avg_hit_at_5", value=avg)

def avg_hit_at_20(*, item_results, **kwargs):
    vals = [
        ev.value for result in item_results
        for ev in result.evaluations
        if ev.name == "hit_at_20"
    ]
    if not vals:
        return Evaluation(name="avg_hit_at_20", value=None)
    avg = sum(vals) / len(vals)
    return Evaluation(name="avg_hit_at_20", value=avg)

def avg_mrr_doc(*, item_results, **kwargs):
    vals = [
        ev.value for result in item_results
        for ev in result.evaluations
        if ev.name == "mrr_doc"
    ]
    if not vals:
        return Evaluation(name="avg_mrr_doc", value=None)
    avg = sum(vals) / len(vals)
    return Evaluation(name="avg_mrr_doc", value=avg)

def avg_nb_correct_doc_in_chunks(*, item_results, **kwargs):
    vals = [
        ev.value for result in item_results
        for ev in result.evaluations
        if ev.name == "nb_correct_doc_in_chunks"
    ]
    if not vals:
        return Evaluation(name="avg_nb_correct_doc_in_chunks", value=None)
    avg = sum(vals) / len(vals)
    return Evaluation(name="avg_nb_correct_doc_in_chunks", value=avg)

In [72]:
dataset = langfuse.get_dataset(dataset_name)
run_name = f"{COLLECTION_NAME}_{datetime.now().isoformat()}"

result = dataset.run_experiment(
    name=COLLECTION_NAME,
    run_name=run_name,
    description=f"Experiment to score retrieval task on dev dataset for {COLLECTION_NAME} index",
    task=retrieval_task,
    evaluators=[hit_at_1_evaluator, hit_at_5_evaluator, hit_at_20_evaluator, mrr_doc_evaluator, nb_correct_doc_in_chunks],
    run_evaluators=[avg_hit_at_1, avg_hit_at_5, avg_hit_at_20, avg_mrr_doc, avg_nb_correct_doc_in_chunks],
    metadata={
        "collection_name": COLLECTION_NAME,
        "top_k": K,
    }
)

print(result.format())

Individual Results: Hidden (13 items)\n💡 Set include_item_results=True to view them\n\n──────────────────────────────────────────────────\n🧪 Experiment: tika_bge_RCTS_window_hybrid
📋 Run name: tika_bge_RCTS_window_hybrid_2026-04-08T15:22:36.128953 - Experiment to score retrieval task on dev dataset for tika_bge__RCTS_1000_200_hybrid_improved_chunking index\n13 items\nEvaluations:\n  • hit_at_1\n  • hit_at_20\n  • nb_correct_doc_in_chunks\n  • hit_at_5\n  • mrr_doc\n\nAverage Scores:\n  • hit_at_1: 0.000\n  • hit_at_20: 0.462\n  • nb_correct_doc_in_chunks: 7.538\n  • hit_at_5: 0.385\n  • mrr_doc: 0.187\n\nRun Evaluations:\n  • avg_hit_at_1: 0.000\n  • avg_hit_at_5: 0.385\n  • avg_hit_at_20: 0.462\n  • avg_mrr_doc: 0.187\n  • avg_nb_correct_doc_in_chunks: 7.538\n\n🔗 Dataset Run:\n   http://localhost:3000/project/cmmvqmjqn0007nw07hbqsche1/datasets/cmnhh6oc2000unw06yp3iv7fv/runs/29d114a1-bf3f-4698-9a36-23785b4ccdd1
